In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score



In [2]:
df = pd.read_csv("/Users/purikunsrinor/Project/Myself/kaggle practice/credit-risk-model/data/processed/cs-training_cleaned.csv", index_col=0)

X = df.drop("SeriousDlqin2yrs", axis = 1)
y = df["SeriousDlqin2yrs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("RRRR")

RRRR


In [3]:
depths = [2, 3, 4, 5, 6, 8, 10, 15]
results = []

for d in depths:
    model = XGBClassifier(
        n_estimators=100,
        max_depth=d,
        random_state=42,
        eval_metric='auc',
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    results.append({'depth': d, 'train_auc': train_auc, 'val_auc': val_auc})
    print(f"depth={d:2d} | train AUC: {train_auc:.4f} | val AUC: {val_auc:.4f}")

depth= 2 | train AUC: 0.8689 | val AUC: 0.8685
depth= 3 | train AUC: 0.8754 | val AUC: 0.8678
depth= 4 | train AUC: 0.8839 | val AUC: 0.8662
depth= 5 | train AUC: 0.8962 | val AUC: 0.8637
depth= 6 | train AUC: 0.9131 | val AUC: 0.8597
depth= 8 | train AUC: 0.9557 | val AUC: 0.8488
depth=10 | train AUC: 0.9865 | val AUC: 0.8363
depth=15 | train AUC: 0.9999 | val AUC: 0.8175


In [9]:
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5]
for lr in learning_rates:
    model = XGBClassifier(
        n_estimators=100,
        learning_rate=lr,
        max_depth=3,
        random_state=42,
        eval_metric='auc',
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    print(f"lr={lr:.3f} | train AUC: {train_auc:.4f} | val AUC: {val_auc:.4f}")

lr=0.001 | train AUC: 0.8317 | val AUC: 0.8323
lr=0.010 | train AUC: 0.8451 | val AUC: 0.8468
lr=0.050 | train AUC: 0.8645 | val AUC: 0.8657
lr=0.100 | train AUC: 0.8690 | val AUC: 0.8685
lr=0.300 | train AUC: 0.8754 | val AUC: 0.8678
lr=0.500 | train AUC: 0.8796 | val AUC: 0.8659


In [11]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_dist = {
    'n_estimators': [100, 300, 500],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 5, 10, 13]
}

print("RRReady")

RRReady


In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb = XGBClassifier(
    random_state=42,
    eval_metric='auc',
    n_jobs=-1
)

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    verbose=1
)

search.fit(X_train, y_train)
print(f"Best CV AUC: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best CV AUC: 0.8649
Best params: {'subsample': 0.9, 'scale_pos_weight': 10, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


In [13]:
best_model = search.best_estimator_
val_pred = best_model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_pred)
print(f"Tuned XGBoost val AUC: {val_auc:.4f}")
print(f"Baseline XGBoost val AUC: 0.8597")
print(f"Improvement: {val_auc - 0.8597:+.4f}")

Tuned XGBoost val AUC: 0.8693
Baseline XGBoost val AUC: 0.8597
Improvement: +0.0096
